# 03 — Analyse exploratoire des données ExpenseAI

## Objectif et périmètre

Ce notebook analyse les **lignes de dépense normalisées dans PostgreSQL**. Il ne lit ni le fichier Excel brut, ni un export CSV, et n'utilise jamais la table `staging_expenses_raw`.

L'unité d'analyse est une ligne de dépense. La variable cible `target` représente la décision observée :

- `0` : dépense **approuvée** ;
- `1` : dépense **refusée**.

L'objectif est descriptif : comprendre les distributions, la saisonnalité et les associations avec la cible. Aucune relation observée ne doit être interprétée comme une causalité. Aucun modèle n'est entraîné dans ce notebook.

## 1. Imports et configuration

In [2]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from scipy.stats import chi2_contingency, mannwhitneyu
from sqlalchemy import text

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
pio.templates.default = "plotly_white"

# Le chemin est résolu aussi bien depuis la racine que depuis notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.utils.data_loader import EXPENSES_ANALYTICS_SQL
from database.connection import create_db_engine

## 2. Chargement depuis PostgreSQL

La requête joint `expenses` aux référentiels `expense_types` et `projects`. Le code `SANS_PROJET` est créé uniquement à la lecture avec `COALESCE` : il ne correspond pas à une ligne artificielle du référentiel `projects`.

In [3]:
engine = create_db_engine()
try:
    with engine.connect() as connection:
        # pandas.read_sql utilise la connexion SQLAlchemy existante.
        df = pd.read_sql(EXPENSES_ANALYTICS_SQL, connection)
        project_controls = pd.read_sql(
            text(
                """
                SELECT
                    COUNT(*) AS project_reference_count,
                    COUNT(*) FILTER (WHERE code = 'SANS_PROJET') AS sentinel_count
                FROM projects
                """
            ),
            connection,
        )
finally:
    engine.dispose()

df["expense_date"] = pd.to_datetime(df["expense_date"], errors="coerce")
df["amount_ttc"] = pd.to_numeric(df["amount_ttc"], errors="coerce")
df["tax_rate"] = pd.to_numeric(df["tax_rate"], errors="coerce")
df["target"] = pd.to_numeric(df["target"], errors="coerce").astype("Int64")
df["billable"] = df["billable"].astype("boolean")
df["status"] = df["target"].map({0: "Approuvée", 1: "Refusée"})

print("Chargement PostgreSQL terminé.")

Chargement PostgreSQL terminé.


## 3. Contrôles de cohérence et vue d'ensemble

In [4]:
target_counts = df["target"].value_counts().sort_index()
controls = pd.Series(
    {
        "Nombre de lignes": len(df),
        "Nombre de colonnes chargées": len(df.columns) - 1,  # status est dérivée
        "Dépenses approuvées": int(target_counts.get(0, 0)),
        "Dépenses refusées": int(target_counts.get(1, 0)),
        "Date minimale": df["expense_date"].min().date(),
        "Date maximale": df["expense_date"].max().date(),
        "Types distincts": df["expense_type"].nunique(),
        "Projets réels dans le référentiel": int(project_controls.loc[0, "project_reference_count"]),
        "Lignes SANS_PROJET": int(df["project_code"].eq("SANS_PROJET").sum()),
        "Valeurs SANS_PROJET dans projects": int(project_controls.loc[0, "sentinel_count"]),
        "Groupes de notes de frais": df["expense_group"].nunique(),
    },
    name="Valeur",
)
display(controls.to_frame())

assert len(df) == 7070, "Le volume normalisé attendu est de 7 070 lignes."
assert target_counts.to_dict() == {0: 6956, 1: 114}, "Répartition cible inattendue."
assert project_controls.loc[0, "sentinel_count"] == 0, "SANS_PROJET ne doit pas exister dans projects."
assert set(df["target"].dropna().unique()) == {0, 1}, "La cible contient une valeur non prévue."
assert not df["expense_date"].isna().any(), "Une date normalisée est invalide."

,Valeur
Nombre de lignes,7070
Nombre de colonnes chargées,8
Dépenses approuvées,6956
Dépenses refusées,114
Date minimale,2025-07-22
Date maximale,2026-07-17
Types distincts,33
Projets réels dans le référentiel,194
Lignes SANS_PROJET,4613
Valeurs SANS_PROJET dans projects,0


In [5]:
quality = pd.DataFrame(
    {
        "type pandas": df.dtypes.astype(str),
        "valeurs non nulles": df.notna().sum(),
        "valeurs manquantes": df.isna().sum(),
        "valeurs uniques": df.nunique(dropna=True),
    }
)
display(quality)

# Aperçu sans l'identifiant de regroupement afin de ne pas exposer d'identifiant métier.
display(df.drop(columns=["expense_group"]).head())

,type pandas,valeurs non nulles,valeurs manquantes,valeurs uniques
expense_group,object,7070,0,6253
expense_date,datetime64[ns],7070,0,343
expense_type,object,7070,0,33
amount_ttc,float64,7070,0,2737
billable,boolean,7070,0,2
project_code,object,7070,0,195
tax_rate,float64,7070,0,20
target,Int64,7070,0,2
status,object,7070,0,2


,expense_date,expense_type,amount_ttc,billable,project_code,tax_rate,target,status
0,2025-07-22,Equipement Télétravail,39.990,False,SANS_PROJET,0.200,1,Refusée
1,2025-07-22,Equipement Télétravail,34.770,False,SANS_PROJET,0.200,1,Refusée
2,2025-07-22,Déjeuner,54.000,False,SANS_PROJET,0.100,0,Approuvée
3,2025-07-22,Boissons avec alcool,18.000,False,SANS_PROJET,0.200,0,Approuvée
4,2025-07-22,Boissons avec alcool,27.000,False,COMMERCE,0.200,0,Approuvée


## 4. Indicateurs clés de synthèse

In [6]:
kpis = pd.DataFrame(
    {
        "Indicateur": [
            "Lignes de dépense",
            "Groupes de notes de frais",
            "Montant TTC total (€)",
            "Montant TTC moyen (€)",
            "Taux d'approbation (%)",
            "Taux de refus (%)",
            "Période couverte (jours)",
        ],
        "Valeur": [
            len(df),
            df["expense_group"].nunique(),
            df["amount_ttc"].sum(),
            df["amount_ttc"].mean(),
            df["target"].eq(0).mean() * 100,
            df["target"].eq(1).mean() * 100,
            (df["expense_date"].max() - df["expense_date"].min()).days + 1,
        ],
    }
)
display(kpis)

,Indicateur,Valeur
0,Lignes de dépense,"7,070.000"
1,Groupes de notes de frais,"6,253.000"
2,Montant TTC total (€),"541,832.670"
3,Montant TTC moyen (€),76.638
4,Taux d'approbation (%),98.388
5,Taux de refus (%),1.612
6,Période couverte (jours),361.000


## 5. Analyse univariée des montants TTC

In [7]:
amount_q1 = df["amount_ttc"].quantile(0.25)
amount_q3 = df["amount_ttc"].quantile(0.75)
amount_iqr = amount_q3 - amount_q1
amount_lower_bound = amount_q1 - 1.5 * amount_iqr
amount_upper_bound = amount_q3 + 1.5 * amount_iqr

amount_report = pd.Series(
    {
        "Minimum": df["amount_ttc"].min(),
        "Premier quartile": amount_q1,
        "Médiane": df["amount_ttc"].median(),
        "Moyenne": df["amount_ttc"].mean(),
        "Troisième quartile": amount_q3,
        "Maximum": df["amount_ttc"].max(),
        "Montants négatifs": df["amount_ttc"].lt(0).sum(),
        "Montants nuls": df["amount_ttc"].eq(0).sum(),
        "Valeurs hors bornes IQR": (
            df["amount_ttc"].lt(amount_lower_bound)
            | df["amount_ttc"].gt(amount_upper_bound)
        ).sum(),
        "Borne IQR basse": amount_lower_bound,
        "Borne IQR haute": amount_upper_bound,
    },
    name="Valeur",
)
display(amount_report.to_frame())

,Valeur
Minimum,-17.700
Premier quartile,13.000
Médiane,32.000
Moyenne,76.638
Troisième quartile,88.000
Maximum,"3,297.000"
Montants négatifs,1.000
Montants nuls,2.000
Valeurs hors bornes IQR,629.000
Borne IQR basse,-99.500


In [8]:
amount_histogram = px.histogram(
    df,
    x="amount_ttc",
    nbins=60,
    title="Distribution des montants TTC — valeurs extrêmes conservées",
    labels={"amount_ttc": "Montant TTC (€)"},
    color_discrete_sequence=["#2563EB"],
)
amount_histogram.update_yaxes(title="Nombre de lignes")
amount_histogram.show()

Les bornes fondées sur l'écart interquartile servent uniquement au **signalement**. Les montants extrêmes, nuls ou négatifs ne sont pas supprimés automatiquement : ils peuvent correspondre à des dépenses valides, des avoirs ou des corrections comptables.

## 6. Analyse univariée des taux de taxe

In [9]:
tax_summary = df["tax_rate"].describe(percentiles=[0.25, 0.5, 0.75]).to_frame("Valeur")
tax_frequencies = (
    df["tax_rate"].value_counts(dropna=False).rename_axis("Taux de taxe").reset_index(name="Nombre")
)
tax_frequencies["Pourcentage"] = tax_frequencies["Nombre"] / len(df) * 100
display(tax_summary)
display(tax_frequencies.head(20))

tax_figure = px.bar(
    tax_frequencies.sort_values("Taux de taxe"),
    x="Taux de taxe",
    y="Nombre",
    title="Fréquence des taux de taxe",
    color_discrete_sequence=["#0F766E"],
)
tax_figure.show()

,Valeur
count,"7,070.000"
mean,0.091
std,0.082
min,0.000
25%,0.000
50%,0.100
75%,0.200
max,0.400


,Taux de taxe,Nombre,Pourcentage
0,0.000,2644,37.397
1,0.200,2022,28.600
2,0.100,1940,27.440
3,0.055,260,3.678
4,0.160,180,2.546
5,0.210,4,0.057
6,0.030,3,0.042
7,0.021,3,0.042
8,0.090,2,0.028
9,0.083,2,0.028


## 7. Analyse univariée des variables catégorielles

In [10]:
def categorical_profile(column: str) -> pd.DataFrame:
    """Construit un profil de fréquence et de cible pour une catégorie."""
    profile = (
        df.groupby(column, dropna=False, observed=True)
        .agg(
            nombre=("target", "size"),
            refus=("target", "sum"),
            taux_refus=("target", "mean"),
            montant_total=("amount_ttc", "sum"),
        )
        .reset_index()
    )
    profile["pourcentage"] = profile["nombre"] / len(df) * 100
    profile["taux_refus"] *= 100
    return profile.sort_values("nombre", ascending=False)


type_profile = categorical_profile("expense_type")
project_profile = categorical_profile("project_code")
billable_profile = categorical_profile("billable")

display(type_profile.head(15))
display(project_profile.head(15))
display(billable_profile)

,expense_type,nombre,refus,taux_refus,montant_total,pourcentage
10,Déjeuner,1379,8,0.580,"151,537.800",19.505
17,Frais kilométriques,1104,0,0.000,"70,101.730",15.615
23,Parking,569,7,1.230,"13,764.770",8.048
29,Taxi / Uber,523,6,1.147,"17,072.670",7.397
0,Abonnement Professionnel,336,8,2.381,"45,605.660",4.752
11,Dîner,333,1,0.300,"31,623.940",4.710
4,Boissons avec alcool,325,1,0.308,"9,446.090",4.597
30,Train,315,3,0.952,"24,800.770",4.455
26,Péage,221,2,0.905,"3,776.300",3.126
2,Abonnements,218,7,3.211,"12,952.660",3.083


,project_code,nombre,refus,taux_refus,montant_total,pourcentage
152,SANS_PROJET,4613,83,1.799,"346,824.370",65.248
128,MANAGEMENT,364,1,0.275,"42,505.780",5.149
59,COMMERCE_,224,2,0.893,"13,275.820",3.168
58,COMMERCE,204,5,2.451,"12,702.040",2.885
125,MAC2412CDANAN0156,123,0,0.000,"7,219.430",1.740
127,MAC2601MWANAN0194,69,0,0.000,"3,702.720",0.976
126,MAC2510PSFNAN0186,58,0,0.000,"3,229.480",0.820
151,ROE2501CUH5624,47,0,0.000,487.700,0.665
66,DEC2406EMUPAR1592,46,0,0.000,"3,165.340",0.651
119,LOR2507HCHPAR1864,44,2,4.545,684.160,0.622


,billable,nombre,refus,taux_refus,montant_total,pourcentage
0,False,6392,108,1.690,"472,215.060",90.410
1,True,678,6,0.885,"69,617.610",9.590


In [11]:
top_types = type_profile.head(15).sort_values("nombre")
type_figure = px.bar(
    top_types,
    x="nombre",
    y="expense_type",
    orientation="h",
    text="nombre",
    title="15 types de dépense les plus fréquents",
    labels={"nombre": "Nombre de lignes", "expense_type": "Type de dépense"},
    color_discrete_sequence=["#2563EB"],
)
type_figure.show()

## 8. Analyse temporelle

In [12]:
# Toutes les variables temporelles sont dérivées de la vraie date de dépense.
df["annee"] = df["expense_date"].dt.year
df["mois"] = df["expense_date"].dt.month
df["jour"] = df["expense_date"].dt.day
df["jour_semaine"] = df["expense_date"].dt.dayofweek
df["trimestre"] = df["expense_date"].dt.quarter
df["est_weekend"] = df["jour_semaine"].isin([5, 6])
df["month"] = df["expense_date"].dt.to_period("M").dt.to_timestamp()

monthly = (
    df.groupby("month", as_index=False)
    .agg(
        nombre=("target", "size"),
        montant_total=("amount_ttc", "sum"),
        montant_moyen=("amount_ttc", "mean"),
        refus=("target", "sum"),
        taux_refus=("target", "mean"),
    )
)
monthly["taux_refus"] *= 100
display(monthly)

,month,nombre,montant_total,montant_moyen,refus,taux_refus
0,2025-07-01,173,"12,995.970",75.121,3,1.734
1,2025-08-01,281,"17,885.980",63.651,5,1.779
2,2025-09-01,697,"51,116.890",73.338,14,2.009
3,2025-10-01,863,"63,420.960",73.489,17,1.970
4,2025-11-01,741,"51,319.250",69.257,7,0.945
5,2025-12-01,652,"56,186.240",86.175,6,0.920
6,2026-01-01,570,"46,413.250",81.427,16,2.807
7,2026-02-01,631,"57,216.630",90.676,5,0.792
8,2026-03-01,711,"50,941.420",71.648,13,1.828
9,2026-04-01,669,"53,360.320",79.761,16,2.392


In [13]:
temporal_figure = go.Figure()
temporal_figure.add_trace(
    go.Scatter(x=monthly["month"], y=monthly["nombre"], mode="lines+markers", name="Volume")
)
temporal_figure.update_layout(
    title="Évolution mensuelle du volume de lignes de dépense",
    xaxis_title="Mois",
    yaxis_title="Nombre de lignes",
    hovermode="x unified",
)
temporal_figure.show()

Note : Les mois de juillet 2025 et juillet 2026 sont partiels et ne doivent pas être comparés directement aux mois complets.

In [14]:
amount_temporal_figure = go.Figure()
amount_temporal_figure.add_trace(
    go.Bar(x=monthly["month"], y=monthly["montant_total"], name="Montant total")
)
amount_temporal_figure.add_trace(
    go.Scatter(
        x=monthly["month"],
        y=monthly["montant_moyen"],
        name="Montant moyen",
        yaxis="y2",
        mode="lines+markers",
        line={"color": "#DC2626"},
    )
)
amount_temporal_figure.update_layout(
    title="Montants TTC mensuels : total et moyenne",
    xaxis_title="Mois",
    yaxis={"title": "Montant TTC total (€)"},
    yaxis2={"title": "Montant TTC moyen (€)", "overlaying": "y", "side": "right"},
    hovermode="x unified",
)
amount_temporal_figure.show()

In [15]:
monthly_status = (
    df.groupby(["month", "status"], observed=True)
    .size()
    .rename("nombre")
    .reset_index()
)
status_temporal_figure = px.line(
    monthly_status,
    x="month",
    y="nombre",
    color="status",
    markers=True,
    title="Évolution mensuelle des statuts",
    labels={"month": "Mois", "nombre": "Nombre de lignes", "status": "Statut"},
    color_discrete_map={"Approuvée": "#2563EB", "Refusée": "#DC2626"},
)
status_temporal_figure.show()

refusal_temporal_figure = px.line(
    monthly,
    x="month",
    y="taux_refus",
    markers=True,
    title="Évolution mensuelle du taux de refus",
    labels={"month": "Mois", "taux_refus": "Taux de refus (%)"},
    color_discrete_sequence=["#DC2626"],
)
refusal_temporal_figure.show()

## 9. Analyse bivariée par rapport à la cible

In [16]:
amount_by_status = (
    df.groupby("status", observed=True)["amount_ttc"]
    .describe(percentiles=[0.25, 0.5, 0.75])
)
display(amount_by_status)

amount_status_figure = px.box(
    df,
    x="status",
    y="amount_ttc",
    color="status",
    points="outliers",
    title="Distribution du montant TTC selon le statut",
    labels={"status": "Statut", "amount_ttc": "Montant TTC (€)"},
    color_discrete_map={"Approuvée": "#2563EB", "Refusée": "#DC2626"},
)
amount_status_figure.update_layout(showlegend=False)
amount_status_figure.show()

,count,mean,std,min,25%,50%,75%,max
status,,,,,,,,
Approuvée,"6,956.000",76.550,144.650,-17.700,12.988,31.990,88.000,"3,297.000"
Refusée,114.000,82.014,94.737,3.000,18.675,41.475,119.463,483.840


In [17]:
MIN_CATEGORY_COUNT = 20

type_target = categorical_profile("expense_type")
project_target = categorical_profile("project_code")
type_target_reliable = type_target[type_target["nombre"] >= MIN_CATEGORY_COUNT].copy()
project_target_reliable = project_target[project_target["nombre"] >= MIN_CATEGORY_COUNT].copy()

display(type_target_reliable.sort_values("taux_refus", ascending=False).head(15))
display(project_target_reliable.sort_values("taux_refus", ascending=False).head(15))

type_refusal_figure = px.bar(
    type_target_reliable.sort_values("taux_refus").tail(15),
    x="taux_refus",
    y="expense_type",
    orientation="h",
    text="nombre",
    title="Taux de refus par type — catégories d'au moins 20 lignes",
    labels={"taux_refus": "Taux de refus (%)", "expense_type": "Type de dépense", "nombre": "Effectif"},
    color_discrete_sequence=["#DC2626"],
)
type_refusal_figure.show()

,expense_type,nombre,refus,taux_refus,montant_total,pourcentage
13,Equipement Télétravail,103,20,19.417,"10,879.020",1.457
1,Abonnement téléphonique,163,15,9.202,"4,029.980",2.306
16,Formations,40,3,7.500,"3,662.020",0.566
24,Petit équipement/petit matériel,104,7,6.731,"12,705.880",1.471
9,Certifications,122,5,4.098,"17,848.460",1.726
20,Hôtel / Appart Hôtel / AirBNB,129,5,3.876,"31,015.180",1.825
31,Transport en commun,190,7,3.684,"2,408.690",2.687
5,Boissons sans alcool,88,3,3.409,"1,408.310",1.245
2,Abonnements,218,7,3.211,"12,952.660",3.083
0,Abonnement Professionnel,336,8,2.381,"45,605.660",4.752


,project_code,nombre,refus,taux_refus,montant_total,pourcentage
189,TIF2503DATPAR1796,22,7,31.818,"1,180.200",0.311
50,CERTIFICATION,29,3,10.345,"4,371.730",0.410
145,RECRUTEMENT,29,2,6.897,"1,027.540",0.410
119,LOR2507HCHPAR1864,44,2,4.545,684.160,0.622
48,CAN2504DHAPAR1791,37,1,2.703,724.280,0.523
60,CONF_PART,40,1,2.500,"5,371.600",0.566
58,COMMERCE,204,5,2.451,"12,702.040",2.885
152,SANS_PROJET,4613,83,1.799,"346,824.370",65.248
59,COMMERCE_,224,2,0.893,"13,275.820",3.168
128,MANAGEMENT,364,1,0.275,"42,505.780",5.149


In [18]:
billable_target = categorical_profile("billable")
tax_target = (
    df.groupby("tax_rate", observed=True)
    .agg(nombre=("target", "size"), refus=("target", "sum"), taux_refus=("target", "mean"))
    .reset_index()
)
tax_target["taux_refus"] *= 100
display(billable_target)
display(tax_target.sort_values("nombre", ascending=False))

,billable,nombre,refus,taux_refus,montant_total,pourcentage
0,False,6392,108,1.690,"472,215.060",90.410
1,True,678,6,0.885,"69,617.610",9.590


,tax_rate,nombre,refus,taux_refus
0,0.000,2644,53,2.005
15,0.200,2022,47,2.324
12,0.100,1940,13,0.670
4,0.055,260,0,0.000
14,0.160,180,1,0.556
16,0.210,4,0,0.000
2,0.021,3,0,0.000
3,0.030,3,0,0.000
7,0.083,2,0,0.000
11,0.090,2,0,0.000


## 10. Tests statistiques exploratoires

Les tests ci-dessous évaluent des **associations** dans l'historique disponible. Ils ne prouvent pas une relation causale. Pour chaque test du khi-deux, les effectifs théoriques sont contrôlés : l'interprétation de la p-valeur n'est retenue que si aucun effectif théorique n'est inférieur à 1 et si au plus 20 % sont inférieurs à 5.

In [19]:
def chi_square_diagnostic(category: str) -> dict:
    """Calcule un khi-deux et vérifie ses conditions usuelles d'application."""
    contingency = pd.crosstab(df[category], df["target"])
    chi2, p_value, degrees, expected = chi2_contingency(contingency)
    expected = np.asarray(expected)
    share_below_five = float((expected < 5).mean())
    conditions_met = bool(expected.min() >= 1 and share_below_five <= 0.20)
    return {
        "variable": category,
        "statistique_chi2": chi2,
        "degres_liberte": degrees,
        "p_value": p_value,
        "effectif_theorique_min": expected.min(),
        "part_effectifs_theoriques_inferieurs_5": share_below_five,
        "conditions_respectees": conditions_met,
    }


chi_type = chi_square_diagnostic("expense_type")
chi_billable = chi_square_diagnostic("billable")
chi_results = pd.DataFrame([chi_type, chi_billable])
display(chi_results)

for result in (chi_type, chi_billable):
    label = "type de dépense" if result["variable"] == "expense_type" else "caractère facturable"
    if not result["conditions_respectees"]:
        interpretation = (
            f"**{label.capitalize()} et cible :** les conditions du khi-deux ne sont pas "
            "suffisamment respectées. La p-valeur est fournie à titre diagnostique, sans "
            "conclusion statistique."
        )
    elif result["p_value"] < 0.05:
        interpretation = (
            f"**{label.capitalize()} et cible :** une association statistique est détectée "
            f"au seuil de 5 % (p = {result['p_value']:.4g}), sans implication causale."
        )
    else:
        interpretation = (
            f"**{label.capitalize()} et cible :** aucune association statistique n'est "
            f"mise en évidence au seuil de 5 % (p = {result['p_value']:.4g})."
        )
    display(Markdown(interpretation))

,variable,statistique_chi2,degres_liberte,p_value,effectif_theorique_min,part_effectifs_theoriques_inferieurs_5,conditions_respectees
0,expense_type,394.216,32,0.000,0.048,0.455,False
1,billable,2.020,1,0.155,10.932,0.000,True


**Type de dépense et cible :** les conditions du khi-deux ne sont pas suffisamment respectées. La p-valeur est fournie à titre diagnostique, sans conclusion statistique.

**Caractère facturable et cible :** aucune association statistique n'est mise en évidence au seuil de 5 % (p = 0.1552).

In [20]:
approved_amounts = df.loc[df["target"].eq(0), "amount_ttc"].dropna()
refused_amounts = df.loc[df["target"].eq(1), "amount_ttc"].dropna()
mann_whitney = mannwhitneyu(
    approved_amounts,
    refused_amounts,
    alternative="two-sided",
)

mann_whitney_result = pd.Series(
    {
        "Statistique U": mann_whitney.statistic,
        "p-valeur": mann_whitney.pvalue,
        "Médiane approuvée (€)": approved_amounts.median(),
        "Médiane refusée (€)": refused_amounts.median(),
        "Effectif approuvé": len(approved_amounts),
        "Effectif refusé": len(refused_amounts),
    },
    name="Valeur",
)
display(mann_whitney_result.to_frame())

if mann_whitney.pvalue < 0.05:
    mw_interpretation = (
        "Le test de Mann–Whitney met en évidence une différence de distribution "
        "des montants entre statuts au seuil de 5 %. Ce résultat est associatif et "
        "ne démontre pas que le montant provoque la décision."
    )
else:
    mw_interpretation = (
        "Le test de Mann–Whitney ne met pas en évidence de différence de distribution "
        "des montants entre statuts au seuil de 5 %."
    )
display(Markdown(mw_interpretation))

,Valeur
Statistique U,"354,676.000"
p-valeur,0.053
Médiane approuvée (€),31.990
Médiane refusée (€),41.475
Effectif approuvé,"6,956.000"
Effectif refusé,114.000


Le test de Mann–Whitney ne met pas en évidence de différence de distribution des montants entre statuts au seuil de 5 %.

## 11. Déséquilibre de la cible

In [21]:
target_distribution = (
    df["status"].value_counts()
    .rename_axis("Statut")
    .reset_index(name="Nombre")
)
target_distribution["Pourcentage"] = target_distribution["Nombre"] / len(df) * 100
display(target_distribution)

target_figure = px.bar(
    target_distribution,
    x="Statut",
    y="Nombre",
    color="Statut",
    text=target_distribution.apply(
        lambda row: f"{int(row['Nombre'])} ({row['Pourcentage']:.2f} %)", axis=1
    ),
    title="Déséquilibre de la variable cible",
    color_discrete_map={"Approuvée": "#2563EB", "Refusée": "#DC2626"},
)
target_figure.update_layout(showlegend=False)
target_figure.update_traces(textposition="outside")
target_figure.show()

,Statut,Nombre,Pourcentage
0,Approuvée,6956,98.388
1,Refusée,114,1.612


Ce déséquilibre impose de toujours présenter les **effectifs**, le **nombre de refus** et le **taux de refus** ensemble. Une catégorie comptant peu d'observations peut afficher un taux instable. Pour une future modélisation, l'accuracy seule serait insuffisante, mais aucune étape de modélisation ou de rééquilibrage n'est réalisée ici.

## 12. Conclusion générale de l'EDA

In [22]:
best_type = type_target_reliable.sort_values(["taux_refus", "nombre"], ascending=False).iloc[0]
best_project = project_target_reliable.sort_values(["taux_refus", "nombre"], ascending=False).iloc[0]
peak_month = monthly.loc[monthly["nombre"].idxmax()]
highest_refusal_month = monthly.loc[monthly["taux_refus"].idxmax()]

chi_type_sentence = (
    "Les conditions du khi-deux ne sont pas respectées pour le type de dépense ; "
    "sa p-valeur ne doit donc pas être interprétée."
    if not chi_type["conditions_respectees"]
    else (
        "Une association statistique entre type et cible est détectée, sans causalité."
        if chi_type["p_value"] < 0.05
        else "Aucune association statistique entre type et cible n'est détectée au seuil de 5 %."
    )
)
chi_billable_sentence = (
    "Les conditions du khi-deux ne sont pas respectées pour le caractère facturable."
    if not chi_billable["conditions_respectees"]
    else (
        "Une association statistique entre caractère facturable et cible est détectée, sans causalité."
        if chi_billable["p_value"] < 0.05
        else "Aucune association statistique entre caractère facturable et cible n'est détectée au seuil de 5 %."
    )
)

conclusion = f"""
### Synthèse factuelle

- La base analytique contient **{len(df):,} lignes** réparties entre **{df['expense_group'].nunique():,} groupes**, sur la période du **{df['expense_date'].min():%d/%m/%Y}** au **{df['expense_date'].max():%d/%m/%Y}**.
- La cible est très déséquilibrée : **{int(target_counts.get(0, 0)):,} approbations ({df['target'].eq(0).mean() * 100:.2f} %)** contre **{int(target_counts.get(1, 0)):,} refus ({df['target'].eq(1).mean() * 100:.2f} %)**.
- Le montant TTC médian est de **{df['amount_ttc'].median():.2f} €** et la moyenne de **{df['amount_ttc'].mean():.2f} €**. Le maximum atteint **{df['amount_ttc'].max():.2f} €** ; **{int(df['amount_ttc'].lt(0).sum())}** montant négatif et **{int(df['amount_ttc'].eq(0).sum())}** montants nuls doivent rester signalés sans suppression automatique.
- Le mois au volume le plus élevé est **{peak_month['month']:%Y-%m}** avec **{int(peak_month['nombre'])} lignes**. Le taux de refus mensuel maximal est observé en **{highest_refusal_month['month']:%Y-%m}** avec **{highest_refusal_month['taux_refus']:.2f} %** ; ce taux doit être lu avec son effectif mensuel.
- Parmi les types comptant au moins {MIN_CATEGORY_COUNT} lignes, **{best_type['expense_type']}** présente le taux de refus descriptif le plus élevé (**{best_type['taux_refus']:.2f} %, n = {int(best_type['nombre'])}**). Parmi les projets répondant au même seuil, **{best_project['project_code']}** atteint **{best_project['taux_refus']:.2f} % (n = {int(best_project['nombre'])})**. Ces écarts sont descriptifs et ne prouvent aucune causalité.
- {chi_type_sentence} {chi_billable_sentence} Le test de Mann–Whitney donne une p-valeur de **{mann_whitney.pvalue:.4g}** ; {mw_interpretation[0].lower() + mw_interpretation[1:]}

### Points d'attention pour la suite

- conserver `expense_group` uniquement comme identifiant de regroupement et respecter les groupes lors d'un futur découpage train/test ;
- ne jamais utiliser les colonnes postérieures à la décision ni les identifiants comme variables prédictives ;
- traiter le fort déséquilibre de cible avec des métriques adaptées, exclusivement dans une future phase de modélisation ;
- encapsuler ultérieurement les transformations dans un pipeline scikit-learn, sans réutiliser les données de test pour apprendre le preprocessing.

Ce notebook s'arrête volontairement à l'analyse exploratoire : aucun modèle, split, rééquilibrage ou hyperparamétrage n'a été réalisé.
""".replace(",", " ")
display(Markdown(conclusion))


### Synthèse factuelle

- La base analytique contient **7 070 lignes** réparties entre **6 253 groupes**  sur la période du **22/07/2025** au **17/07/2026**.
- La cible est très déséquilibrée : **6 956 approbations (98.39 %)** contre **114 refus (1.61 %)**.
- Le montant TTC médian est de **32.00 €** et la moyenne de **76.64 €**. Le maximum atteint **3297.00 €** ; **1** montant négatif et **2** montants nuls doivent rester signalés sans suppression automatique.
- Le mois au volume le plus élevé est **2025-10** avec **863 lignes**. Le taux de refus mensuel maximal est observé en **2026-01** avec **2.81 %** ; ce taux doit être lu avec son effectif mensuel.
- Parmi les types comptant au moins 20 lignes  **Equipement Télétravail** présente le taux de refus descriptif le plus élevé (**19.42 %  n = 103**). Parmi les projets répondant au même seuil  **TIF2503DATPAR1796** atteint **31.82 % (n = 22)**. Ces écarts sont descriptifs et ne prouvent aucune causalité.
- Les conditions du khi-deux ne sont pas respectées pour le type de dépense ; sa p-valeur ne doit donc pas être interprétée. Aucune association statistique entre caractère facturable et cible n'est détectée au seuil de 5 %. Le test de Mann–Whitney donne une p-valeur de **0.05306** ; le test de Mann–Whitney ne met pas en évidence de différence de distribution des montants entre statuts au seuil de 5 %.

### Points d'attention pour la suite

- conserver `expense_group` uniquement comme identifiant de regroupement et respecter les groupes lors d'un futur découpage train/test ;
- ne jamais utiliser les colonnes postérieures à la décision ni les identifiants comme variables prédictives ;
- traiter le fort déséquilibre de cible avec des métriques adaptées  exclusivement dans une future phase de modélisation ;
- encapsuler ultérieurement les transformations dans un pipeline scikit-learn  sans réutiliser les données de test pour apprendre le preprocessing.

Ce notebook s'arrête volontairement à l'analyse exploratoire : aucun modèle  split  rééquilibrage ou hyperparamétrage n'a été réalisé.
